# **MamaCare-AI Maternal Triage Training Pipeline**

This notebook contains the complete, self-contained training and deployment pipeline for the maternal health triage classifier using standard Hugging Face `Trainer` and `distilbert-base-multilingual-cased`.

### **Step 1: Install Environment Dependencies**
Installs the required Hugging Face libraries (`transformers`, `datasets`, and `accelerate`).

In [ ]:
# Install the required Hugging Face libraries
!pip install -q transformers datasets accelerate
print("Libraries installed successfully!")

### **Step 2: Ingest and Prepare Few-Shot Anchors**
Loads the clinical danger sign anchors from `/kaggle/input/triage-anchor-maternal/triage_anchors.json` and oversamples them (10x) to give the model a robust signal.

In [ ]:
import json
import pandas as pd
from datasets import Dataset

# 1. Load your uploaded triage_anchors.json from Kaggle input path
print("Loading custom triage anchors...")
with open("/kaggle/input/triage-anchor-maternal/triage_anchors.json", "r", encoding="utf-8") as f:
    anchors_data = json.load(f)

# 2. Create DataFrame and keep only text and label columns
df = pd.DataFrame(anchors_data)
df = df[['text', 'label']]

# 3. Oversample the anchors to help standard Trainer learn efficiently
df_oversampled = pd.concat([df] * 10, ignore_index=True)

print(f"\nOriginal Dataset Size: {len(df)} rows")
print(f"Oversampled Dataset Size: {len(df_oversampled)} rows")
print(df_oversampled['label'].value_counts())

### **Step 3: Tokenization & Model Loading**
Loads the multilingual checkpoint, tokenizes the dataset, and loads the sequence classification model with 3 output labels (LOW, MEDIUM, HIGH).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_checkpoint = "distilbert-base-multilingual-cased"
print(f"Loading tokenizer and model for {model_checkpoint}...")

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# Convert pandas DataFrame to Hugging Face Dataset and tokenize
dataset = Dataset.from_pandas(df_oversampled)
tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.shuffle(seed=42)

# Load sequence classification model (3 classes: 0=LOW, 1=MEDIUM, 2=HIGH)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3)

### **Step 4: Run Fine-Tuning**
Configures `TrainingArguments` and trains the model for 3 epochs using the standard `Trainer`.

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    evaluation_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

print("Starting model training...")
trainer.train()
print("Training completed!")

### **Step 5: Test Model Predictions**
Evaluates the model on test phrases to check classifications.

In [ ]:
import torch

# Test predictions
test_phrases = [
    "I am bleeding heavily since this morning",
    "I feel fine, just a bit tired",
    "I have a mild headache but my vision is normal",
    "My feet and hands are swollen and I cannot see clearly",
    "I am fine no issues"
]

model.eval()
labels_map = {0: "LOW", 1: "MEDIUM", 2: "HIGH"}

print("\n--- Test Predictions ---")
for phrase in test_phrases:
    inputs = tokenizer(phrase, return_tensors="pt", truncation=True, padding=True).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=-1).item()
    print(f"Text: '{phrase}' -> Predicted Risk: {labels_map[prediction]}")

### **Step 6: Push Model to Hugging Face Hub**
Prompts login to Hugging Face and pushes model weights and tokenizers to the Hub.

In [ ]:
from huggingface_hub import notebook_login
# This will prompt you to enter your Hugging Face Access Token (Write permission required)
notebook_login()

# Save tokenizer and model locally
model.save_pretrained("./mamacare-triage-model")
tokenizer.save_pretrained("./mamacare-triage-model")

# Push to Hugging Face Hub (Replace "sammydamz" with your actual HF username)
model.push_to_hub("sammydamz/mamacare-triage-model")
tokenizer.push_to_hub("sammydamz/mamacare-triage-model")
print("Model successfully uploaded to Hugging Face Hub!")

### **Step 7: Automated Model Evaluation**
Loads the evaluation test dataset and computes standard metrics (Accuracy, Precision, Recall, F1-Score) to ensure the model has learned the guidelines correctly.

In [ ]:
import json
import torch
from sklearn.metrics import classification_report, accuracy_score

# 1. Load evaluation test data from Kaggle input path
print("Loading evaluation test data...")
with open("/kaggle/input/triage-anchor-maternal/triage_test_data.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_texts = [item["text"] for item in test_data]
true_labels = [item["expected_label"] for item in test_data]
labels_map = {0: "LOW", 1: "MEDIUM", 2: "HIGH"}

# 2. Run predictions
predicted_labels = []
model.eval()

print("Running evaluation inference...")
for text in test_texts:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=-1).item()
    predicted_labels.append(pred)

# 3. Calculate and display metrics
accuracy = accuracy_score(true_labels, predicted_labels)
print(f"\nModel Accuracy on Test Set: {accuracy * 100:.2f}%")

print("\n--- Detailed Classification Report ---")
print(classification_report(true_labels, predicted_labels, target_names=["LOW", "MEDIUM", "HIGH"]))
